# Optional fresh YouTube collection

Historical query strategy preserved for reference. This makes live API requests, consumes quota, and cannot recreate the historical snapshot. Results go to `outputs/fresh_micro_collection.csv`; committed data is preserved. Large-channel collection was not fully supplied.

Set `YOUTUBE_API_KEY` outside the notebook and then change `RUN_COLLECTION` to `True` only when you want fresh data.

In [1]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
RUN_COLLECTION = False

In [2]:
if not RUN_COLLECTION:
    print('Collection disabled. Use the committed snapshots for analysis.')
else:
    if not os.environ.get('YOUTUBE_API_KEY'):
        raise ValueError('Set YOUTUBE_API_KEY in your environment first.')
    (ROOT / 'outputs').mkdir(exist_ok=True)
    # CELL 0
    
    
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    import pandas as pd
    import numpy as np
    import re
    import time
    from sklearn.preprocessing import StandardScaler
    
    # 1) SETUP
    API_KEY = os.environ["YOUTUBE_API_KEY"]
    youtube = build("youtube", "v3", developerKey=API_KEY)
    
    TARGET_ROWS = 1200
    OUTPUT_PATH = str(ROOT / 'outputs/fresh_micro_collection.csv')
    
    queries = [
        "first vlog",
        "small youtuber",
        "study vlog",
        "day in my life",
        "college vlog",
        "new channel intro",
        "my first youtube video",
        "beginner youtuber",
        "small creator vlog",
        "daily routine vlog",
        "what i eat in a day",
        "week in my life",
        "aesthetic vlog",
        "study with me",
        "new vlogger"
    ]
    
    PAGES_PER_QUERY = 2
    
    
    # 2) HELPER FUNCTIONS
    def clean_text(text):
        if text is None:
            return ""
        text = str(text).replace("\n", " ").replace("\r", " ").strip()
        return re.sub(r"\s+", " ", text)
    
    def clean_tags(tags):
        if not tags:
            return ""
        return " ||| ".join(clean_text(tag) for tag in tags)
    
    def extract_hashtags(text):
        if not text:
            return []
        return re.findall(r"#\w+", text)
    
    def safe_int(d, key):
        try:
            return int(d.get(key, 0))
        except Exception:
            return 0
    
    def chunk_list(lst, chunk_size=50):
        for i in range(0, len(lst), chunk_size):
            yield lst[i:i + chunk_size]
    
    def count_joined_hashtags(s):
        if not s or str(s).strip() == "":
            return 0
        return len([x for x in str(s).split(" ||| ") if x.strip() != ""])
    
    
    # 3) STEP 1: COLLECT VIDEO IDS
    print("Step 1: Collecting candidate video IDs...")
    video_candidates = []
    seen_video_ids = set()
    
    try:
        for q in queries:
            next_page_token = None
    
            for _ in range(PAGES_PER_QUERY):
                search_res = youtube.search().list(
                    q=q,
                    part="snippet",
                    type="video",
                    order="date",
                    maxResults=50,
                    pageToken=next_page_token
                ).execute()
    
                for item in search_res.get("items", []):
                    video_id = item["id"].get("videoId")
                    channel_id = item["snippet"].get("channelId")
    
                    if not video_id or not channel_id:
                        continue
    
                    if video_id not in seen_video_ids:
                        seen_video_ids.add(video_id)
                        video_candidates.append({
                            "video_id": video_id,
                            "channel_id": channel_id
                        })
    
                next_page_token = search_res.get("nextPageToken")
                if not next_page_token:
                    break
    
                time.sleep(0.2)
    
    except HttpError as e:
        print("Quota or API error during search step:")
        print("YouTube API request failed; HTTP status:", e.resp.status)
        print("If this says quotaExceeded, stop and run again after quota reset.")
    
    print(f"Candidate videos collected: {len(video_candidates)}")
    
    if len(video_candidates) == 0:
        raise ValueError("No candidate videos were collected. Check API key, quota, or queries.")
    
    
    # 4) STEP 2: BATCH FETCH VIDEO DETAILS
    print("Step 2: Fetching video details in batches...")
    video_ids = [x["video_id"] for x in video_candidates]
    video_detail_rows = []
    
    for batch_num, batch in enumerate(chunk_list(video_ids, 50), start=1):
        try:
            res = youtube.videos().list(
                part="snippet,statistics,contentDetails",
                id=",".join(batch)
            ).execute()
    
            for item in res.get("items", []):
                video_detail_rows.append({
                    "video_id": item.get("id", ""),
                    "video_title": clean_text(item["snippet"].get("title", "")),
                    "description": clean_text(item["snippet"].get("description", "")),
                    "tags": clean_tags(item["snippet"].get("tags", [])),
                    "category_id": str(item["snippet"].get("categoryId", "")),
                    "published_at": item["snippet"].get("publishedAt", ""),
                    "channel_id": item["snippet"].get("channelId", ""),
                    "view_count": safe_int(item.get("statistics", {}), "viewCount"),
                    "like_count": safe_int(item.get("statistics", {}), "likeCount"),
                    "comment_count": safe_int(item.get("statistics", {}), "commentCount"),
                    "duration": item.get("contentDetails", {}).get("duration", "")
                })
    
            if batch_num % 10 == 0:
                print(f"  Processed {batch_num} video batches...")
    
            time.sleep(0.2)
    
        except HttpError as e:
            print("YouTube API request failed; HTTP status:", e.resp.status)
            time.sleep(1)
    
    df_videos = pd.DataFrame(video_detail_rows)
    print(f"Detailed videos fetched: {len(df_videos)}")
    
    if df_videos.empty:
        raise ValueError("No video details were fetched.")
    
    
    # 5) STEP 3: BATCH FETCH CHANNEL DETAILS
    print("Step 3: Fetching channel details in batches...")
    unique_channel_ids = df_videos["channel_id"].dropna().unique().tolist()
    channel_rows = []
    
    for batch_num, batch in enumerate(chunk_list(unique_channel_ids, 50), start=1):
        try:
            res = youtube.channels().list(
                part="snippet,statistics",
                id=",".join(batch)
            ).execute()
    
            for item in res.get("items", []):
                channel_rows.append({
                    "channel_id": item.get("id", ""),
                    "channel_title": clean_text(item["snippet"].get("title", "")),
                    "subscriber_count": safe_int(item.get("statistics", {}), "subscriberCount")
                })
    
            if batch_num % 10 == 0:
                print(f"  Processed {batch_num} channel batches...")
    
            time.sleep(0.2)
    
        except HttpError as e:
            print("YouTube API request failed; HTTP status:", e.resp.status)
            time.sleep(1)
    
    df_channels = pd.DataFrame(channel_rows)
    print(f"Channels fetched: {len(df_channels)}")
    
    if df_channels.empty:
        raise ValueError("No channel details were fetched.")
    
    
    # 6) STEP 4: MERGE AND FILTER MICRO CREATORS
    print("Step 4: Merging and filtering creators under 10K subscribers...")
    df_micro = df_videos.merge(df_channels, on="channel_id", how="left")
    df_micro = df_micro.drop_duplicates(subset=["video_id"]).reset_index(drop=True)
    df_micro = df_micro[df_micro["subscriber_count"] < 10000].copy()
    df_micro = df_micro.reset_index(drop=True)
    
    print(f"Micro-creator rows after filter: {len(df_micro)}")
    
    if len(df_micro) == 0:
        raise ValueError("No micro-creators found under 10K subscribers.")
    
    
    # 7) STEP 5: HASHTAGS + LIKE RATE
    print("Step 5: Creating hashtag and engagement features...")
    
    df_micro["hashtags_from_title"] = df_micro["video_title"].apply(
        lambda x: " ||| ".join(extract_hashtags(x))
    )
    df_micro["hashtags_from_desc"] = df_micro["description"].apply(
        lambda x: " ||| ".join(extract_hashtags(x))
    )
    
    df_micro["hashtag_count"] = (
        df_micro["hashtags_from_title"].apply(count_joined_hashtags) +
        df_micro["hashtags_from_desc"].apply(count_joined_hashtags)
    )
    
    df_micro["like_rate"] = np.where(
        df_micro["view_count"] > 0,
        df_micro["like_count"] / df_micro["view_count"],
        0
    )
    
    
    if len(df_micro) > TARGET_ROWS:
        df_micro = df_micro.head(TARGET_ROWS).copy()
    
    print(f"Rows kept for final dataset: {len(df_micro)}")
    
    
    # 8) STEP 6: FETCH TOP COMMENTS
    print("Step 6: Fetching top comments...")
    
    top_comments_col = []
    comment_likes_col = []
    
    for i, video_id in enumerate(df_micro["video_id"], start=1):
        comments_text = []
        comments_likes = []
    
        try:
            res = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=10,
                order="relevance",
                textFormat="plainText"
            ).execute()
    
            for c in res.get("items", []):
                snippet = c["snippet"]["topLevelComment"]["snippet"]
                comments_text.append(clean_text(snippet.get("textDisplay", "")))
                comments_likes.append(str(snippet.get("likeCount", 0)))
    
        except HttpError:
            pass
        except Exception:
            pass
    
        top_comments_col.append(" ||| ".join(comments_text))
        comment_likes_col.append(" ||| ".join(comments_likes))
    
        if i % 100 == 0:
            print(f"  Comments fetched for {i} videos...")
    
        time.sleep(0.1)
    
    df_micro["top_comments_text"] = top_comments_col
    df_micro["comment_like_count"] = comment_likes_col
    
    
    # 9) STEP 7: CLEAN TYPES
    print("Step 7: Standardizing data types...")
    
    df_micro["published_at"] = pd.to_datetime(df_micro["published_at"], errors="coerce")
    
    numeric_cols = [
        "view_count",
        "like_count",
        "comment_count",
        "subscriber_count",
        "hashtag_count",
        "like_rate"
    ]
    
    for col in numeric_cols:
        df_micro[col] = pd.to_numeric(df_micro[col], errors="coerce").fillna(0)
    
    
    # 10) STEP 8: LOG TRANSFORMS
    print("Step 8: Creating log-transformed variables...")
    
    df_micro["log_view_count"] = np.log1p(df_micro["view_count"])
    df_micro["log_like_count"] = np.log1p(df_micro["like_count"])
    df_micro["log_comment_count"] = np.log1p(df_micro["comment_count"])
    df_micro["log_subscriber_count"] = np.log1p(df_micro["subscriber_count"])
    
    
    # 11) STEP 9: SCALING
    print("Step 9: Scaling variables...")
    
    scale_cols = [
        "log_view_count",
        "log_like_count",
        "log_comment_count",
        "log_subscriber_count",
        "like_rate",
        "hashtag_count"
    ]
    
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(df_micro[scale_cols])
    
    df_scaled = pd.DataFrame(
        scaled_values,
        columns=[f"scaled_{c}" for c in scale_cols]
    )
    
    df_micro = pd.concat([df_micro.reset_index(drop=True), df_scaled], axis=1)
    
    
    # 12) FINAL COLUMN ORDER
    print("Step 10: Reordering final columns...")
    
    final_cols = [
        "video_id",
        "video_title",
        "description",
        "tags",
        "category_id",
        "published_at",
        "channel_id",
        "channel_title",
        "view_count",
        "like_count",
        "comment_count",
        "duration",
        "subscriber_count",
        "hashtags_from_title",
        "hashtags_from_desc",
        "hashtag_count",
        "top_comments_text",
        "comment_like_count",
        "like_rate",
        "log_view_count",
        "log_like_count",
        "log_comment_count",
        "log_subscriber_count",
        "scaled_log_view_count",
        "scaled_log_like_count",
        "scaled_log_comment_count",
        "scaled_log_subscriber_count",
        "scaled_like_rate",
        "scaled_hashtag_count"
    ]
    
    df_micro = df_micro[final_cols]
    
    
    # 13) EXPORT
    print("Step 11: Exporting file...")
    df_micro.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    
    print("DONE - dataset created successfully")
    print("Saved file to:", OUTPUT_PATH)
    print("Final rows:", len(df_micro))
    print(df_micro.head())
    
    # CELL 1

Collection disabled. Use the committed snapshots for analysis.
